In [4]:
# IMPORTS Y CONFIGURACIÓN DE RUTAS DE PROYECTO
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import pickle

# IMPORTS DESDE MÓDULO DE CONFIGURACIÓN
from Configuracion import CARGADOS, PROCESADOS, CSV_GENERADOS, IMPLEMENTACION_GOBERNANZA, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS
from Configuracion import RECURSOS

# CREACIÓN DE CARPETAS SEGMENTADAS
modelo_carpetas = [
    CARGADOS, PROCESADOS, CSV_GENERADOS,
    IMPLEMENTACION_GOBERNANZA, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS,
    RECURSOS
]

for modelo_carpeta in modelo_carpetas:
    modelo_carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada correctamente ------")

# LECTURA DEL ARCHIVO .PKL USANDO LA RUTA CONFIGURADA
RUTA_PKL = ANALITICOS_PARQUET / "datos_split_guia8.pkl"

with open(RUTA_PKL, "rb") as f:
    datos = pickle.load(f)

X_train = datos["X_train"]
X_val   = datos["X_val"]
X_test  = datos["X_test"]
y_train = datos["y_train"]
y_val   = datos["y_val"]
y_test  = datos["y_test"]

COLS_NUMERICAS   = datos["COLS_NUMERICAS"]
COLS_CATEGORICAS = datos["COLS_CATEGORICAS"]
TARGET           = datos["TARGET"]

print(f"¡Datos cargados con éxito desde: {RUTA_PKL}!")
print(f"Registros de entrenamiento: {len(X_train)} | Registros de prueba: {len(X_test)}")

Estructura de carpetas creada correctamente ------
¡Datos cargados con éxito desde: ..\Datos\Analiticos\Parquet\datos_split_guia8.pkl!
Registros de entrenamiento: 396267 | Registros de prueba: 84915


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Preprocesador para estandarizar las numéricas y aplicar One-Hot a las categóricas
preprocesador = ColumnTransformer([
    ("num", StandardScaler(), COLS_NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), COLS_CATEGORICAS),
])

# Creamos el pipeline con el Random Forest
modelo_rf = Pipeline([
    ("prep", preprocesador),
    ("clf", RandomForestClassifier(
        n_estimators=100, 
        class_weight="balanced", 
        random_state=42, 
        n_jobs=-1
    )),
])

# Entrenamos con los datos de entrenamiento que se cargaron del archivo
modelo_rf.fit(X_train, y_train)
print("¡Modelo Random Forest entrenado exitosamente!")

¡Modelo Random Forest entrenado exitosamente!


In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Generar predicciones sobre el conjunto de prueba
y_pred_rf = modelo_rf.predict(X_test)
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

# Calcular métricas
metricas_rf = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1-Score": f1_score(y_test, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
}

print("Métricas de tu Modelo Adicional (Random Forest):")
for nombre, valor in metricas_rf.items():
    print(f"{nombre}: {valor:.4f}")

# Matriz de confusión
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()
print(f"\nMatriz de confusión -> TN = {tn} | FP = {fp} | FN = {fn} | TP = {tp}")

Métricas de tu Modelo Adicional (Random Forest):
Accuracy: 0.9969
Precision: 0.9870
Recall: 0.9975
F1-Score: 0.9922
ROC-AUC: 0.9999

Matriz de confusión -> TN = 67970 | FP = 220 | FN = 42 | TP = 16683


In [ ]:
import pandas as pd

# Métricas de referencia del modelo base
metricas_base = {
    "Accuracy": 0.8457,
    "Precision": 0.5644,
    "Recall": 0.9503,
    "F1-Score": 0.7082,
    "ROC-AUC": 0.9558,
}

# Tabla lado a lado
tabla_comparativa = pd.DataFrame({
    "Métrica": list(metricas_rf.keys()),
    "Regresión Logística (Base)": list(metricas_base.values()),
    "Random Forest (Adicional)": list(metricas_rf.values())
})

print("\n--- TABLA COMPARATIVA FINALES ---")
print(tabla_comparativa.to_string(index=False))


--- TABLA COMPARATIVA FINALES ---
  Métrica  Regresión Logística (Base)  Random Forest (Adicional)
 Accuracy                      0.8457                   0.996915
Precision                      0.5644                   0.986985
   Recall                      0.9503                   0.997489
 F1-Score                      0.7082                   0.992209
  ROC-AUC                      0.9558                   0.999866
